<a href="https://colab.research.google.com/github/tanvisht/Data-Science-NYU-Stern/blob/Assignments/Challenge_4_Tanvish.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Tanvish Tuplondhe
##Challenge 4: Data Preparation
IE-GY 9113 SPRING 2026

##PROBLEM 1: Marketing Performance

In [3]:
import pandas as pd
import numpy as np

# Loading the dataset from URL
url = 'https://raw.githubusercontent.com/jcbonilla/Analytics/master/BAData/SocialMediaPerformanceData'
df = pd.read_csv(url)
df.head(5)


,platform,post_id,followers,Impressions,comments,shares,reactions,videoviews,saves,replies,retweets,engagements
0,TikTok,7229050411204857134,30897.0,453945,924,6216,73452,453945,0,0,0,80592
1,TikTok,7270674276812934443,35551.0,511,1,0,21,511,0,0,0,22
2,Instagram,CwTn1P_v09z,102403.0,1241,1,2,87,1241,3,0,0,90
3,TikTok,7270673877955579178,3429.0,637,4,1,39,637,0,0,0,44
4,Instagram,CwS21tJoPED,223544.0,45950,89,9,3367,45950,123,0,0,3465


## Question 1.1: Compute and Identify Outliers

In [4]:
# Define the columns to analyze for outliers based on the question
columns_to_check = ['followers', 'Impressions', 'comments', 'shares']

# Dictionary to store the number of outliers for each column
outlier_counts = {}
total_outliers = 0

for col in columns_to_check:
    # Calculate the 1st (Q1) and 3rd (Q3) quartiles
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # Calculate the Interquartile Range (IQR)
    IQR = Q3 - Q1

    # Define the lower and upper bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter the dataframe to find rows that fall outside these bounds
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    # Count the outliers and store them
    count = len(outliers)
    outlier_counts[col] = count
    total_outliers += count

# Print the results
print("Outliers observed per metric:")
for col, count in outlier_counts.items():
    print(f"- {col}: {count}")

print(f"\nTotal outliers observed across these 4 metrics: {total_outliers}")

Outliers observed per metric:
- followers: 2940
- Impressions: 2866
- comments: 2405
- shares: 2868

Total outliers observed across these 4 metrics: 11079


**Analysis:**

Using the Interquartile Range (IQR) method, I computed and identified a massive number of extreme values within the dataset. Specifically, I observed 2,940 outliers in followers, 2,866 in impressions, 2,405 in comments, and 2,868 in shares.

In total, there are 11,079 outliers across these four metrics. This high volume of outliers indicates that the social media data is heavily right-skewed.

This makes logical sense for marketing performance data, as a small handful of "viral" posts or highly influential accounts will naturally generate exponentially more impressions, comments, and shares compared to the median performance.

## Question 1.2: Select a Normalization Approach

In [5]:
from sklearn.preprocessing import RobustScaler

# Variables to normalize based on the previous question
cols_to_normalize = ['followers', 'Impressions', 'comments', 'shares']

# I am choosing RobustScaler because it uses the median and the Interquartile Range (IQR).
# Since we just found over 11,000 outliers in the previous step, standard methods
# like Min-Max or Z-score would be heavily skewed by those extreme values.
scaler = RobustScaler()

# Create a copy of the dataframe so we don't overwrite the original data
df_normalized = df.copy()

# Fit and transform the data
df_normalized[cols_to_normalize] = scaler.fit_transform(df[cols_to_normalize])

# Display the summary statistics to verify the normalization worked
print("Summary statistics after applying RobustScaler:\n")
print(df_normalized[cols_to_normalize].describe().round(3))

Summary statistics after applying RobustScaler:

       followers  Impressions   comments     shares
count  17570.000    17659.000  17659.000  17659.000
mean       2.033        2.531      4.783     18.735
std        7.628       28.124     64.623    286.501
min       -0.223       -0.179     -0.154     -0.167
25%       -0.196       -0.139     -0.154     -0.167
50%        0.000        0.000      0.000      0.000
75%        0.804        0.861      0.846      0.833
max       55.485     3148.519   3779.615  15714.000


**Analysis:**

For this dataset, I selected the Robust Scaler as my normalization approach. The rationale for this choice stems directly from my findings in the previous step, where I identified over 11,000 outliers across these four metrics. Traditional scaling methods, like Min-Max scaling or standard Z-score normalization, rely heavily on the minimum, maximum, and mean values.

Because social media data is extremely right-skewed (a few viral posts have massive engagement), using those traditional methods would cause the extreme outliers to compress the vast majority of my normal data into a tiny, unusable fraction of the scale.

Robust Scaler is perfect here because it centers the data around the median and scales it using the Interquartile Range (IQR). As my output shows, the median (50%) is exactly 0.000 for all metrics, and the middle 50% of the data is scaled neatly, completely ignoring the pull of the extreme maximums while still keeping those outliers in the dataset for analysis.

## Question 1.3: Determine the Best Platform

In [6]:
# Group the normalized dataframe by 'platform' and calculate the average for each metric
platform_avg = df_normalized.groupby('platform')[['followers', 'Impressions', 'comments', 'shares']].mean()

# To determine the "best" platform overall, we can create a composite score
# by summing the averages of these normalized metrics.
# (Summing them works well since they are all now on a comparable, normalized scale)
platform_avg['composite_score'] = platform_avg.sum(axis=1)

# Sort the platforms by the composite score in descending order to see the winner
best_platforms = platform_avg.sort_values(by='composite_score', ascending=False)

print("Average Normalized Performance by Platform:\n")
print(best_platforms.round(3))

Average Normalized Performance by Platform:

                followers  Impressions  comments  shares  composite_score
platform                                                                 
TikTok              0.221        4.584     9.695  33.720           48.221
Instagram           1.774        1.979     2.723  22.681           29.157
Facebook            7.811        2.369     5.449   6.397           22.026
LinkedIn           -0.200        0.382    -0.025   1.078            1.235
Twitter             0.586        0.619    -0.108  -0.006            1.092
YouTube            -0.184        0.021     0.079   0.858            0.774
Pinterest          -0.210       -0.133    -0.154  -0.162           -0.659
Youtube Shorts     -0.223       -0.175    -0.154  -0.167           -0.718


**Analysis:**

To answer which platform is the best based on the above data, I looked at the composite score calculated from our normalized metrics, and TikTok is the clear winner. It achieved the highest overall score of 48.221, significantly outperforming the runners-up, Instagram (29.157) and Facebook (22.026).



While Facebook actually has a higher normalized follower count (7.811) compared to TikTok (0.221), TikTok vastly dominates in active engagement—specifically in shares (33.720), comments (9.695), and Impressions (4.584). This means that despite having fewer baseline followers on average, the content on TikTok is reaching a wider audience and driving much higher interactive engagement. Therefore, because the goal of marketing performance is active user engagement rather than just a static follower count, TikTok is the best platform.

##PROBLEM 2: Bank loans - missing values

##Question 2.1: Total Missing Values

In [8]:
import pandas as pd

# Load the bank loans dataset
# Fixing the line break from the provided URL
url_loan = 'https://raw.githubusercontent.com/jolineuichanco/DataAnalytics/main/challenges/week4/loan_data.csv'
df_loan = pd.read_csv(url_loan)

# 1. How many missing values are in the dataset?
total_missing = df_loan.isnull().sum().sum()

print(f"Total records loaded: {df_loan.shape[0]}")
print(f"Total columns loaded: {df_loan.shape[1]}")
print(f"Total missing values in the entire dataset: {total_missing}")

Total records loaded: 10000
Total columns loaded: 101
Total missing values in the entire dataset: 288472


**Analysis:**

The dataset contains 10,000 records and 101 columns, giving us a total of 1,010,000 individual data points. Out of these, I identified 288,472 missing values across the entire dataset. This indicates that roughly 28.5% of the total data is missing. A missing data rate this high emphasizes that rigorous data cleaning and imputation are necessary before we can perform any reliable modeling or analysis.

## Question 2.2: Missing Values Per Column

In [9]:
# Calculate the sum of null values for every single column
missing_per_column = df_loan.isnull().sum()

# Since there are 101 columns, let's filter to show only the columns that
# actually have missing values, sorted from most missing to least missing
# for better readability in the output.
missing_counts = missing_per_column[missing_per_column > 0].sort_values(ascending=False)

print("Missing values per column name:")
print(missing_counts.to_string())

Missing values per column name:
COMMONAREA_AVG                  7008
COMMONAREA_MODE.1               7008
COMMONAREA_MODE                 7008
NONLIVINGAPARTMENTS_AVG         6940
NONLIVINGAPARTMENTS_MEDI        6940
NONLIVINGAPARTMENTS_AVG.1       6940
FONDKAPREMONT_MODE              6864
FONDKAPREMONT_MODE.1            6864
LIVINGAPARTMENTS_MODE           6833
LIVINGAPARTMENTS_AVG            6833
FLOORSMIN_MODE                  6793
FLOORSMIN_AVG                   6793
FLOORSMIN_MEDI                  6793
YEARS_BUILD_MODE                6662
YEARS_BUILD_AVG                 6662
YEARS_BUILD_MEDI                6662
YEARS_BUILD_AVG.1               6662
LANDAREA_AVG.1                  5958
LANDAREA_MEDI                   5958
LANDAREA_AVG                    5958
BASEMENTAREA_MODE.1             5858
BASEMENTAREA_MODE               5858
BASEMENTAREA_AVG                5858
EXT_SOURCE_1                    5678
NONLIVINGAREA_MEDI              5532
NONLIVINGAREA_MODE              5532
NONLIV

**Analysis:**

By calculating the missing values per column, I found that 63 out of the 101 columns contain at least some missing data. The distribution is highly uneven: variables related to housing details (like `COMMONAREA_AVG`, `NONLIVINGAPARTMENTS_AVG`, and `FONDKAPREMONT_MODE`) are missing about 70% of their data (around 7,000 out of 10,000 records).

On the other hand, some columns like `AMT_ANNUITY` are only missing a single value. This tells me that while some features will just need minor imputation, a large chunk of the housing-related variables might be too sparse to be useful and will need heavy filtering.

## Question 2.3: Remove rows with more than 45% missing values

In [10]:
# Calculate the proportion of missing values for each row
missing_fraction_per_row = df_loan.isnull().mean(axis=1)

# Keep only the rows where the fraction of missing values is 45% (0.45) or less
df_loan_rows_cleaned = df_loan[missing_fraction_per_row <= 0.45]

# The question specifically asks how many COLUMNS remain after removing ROWS
rows_remaining = df_loan_rows_cleaned.shape[0]
columns_remaining = df_loan_rows_cleaned.shape[1]

print(f"Rows remaining after dropping >45% missing: {rows_remaining}")
print(f"Columns remaining in the dataset: {columns_remaining}")

Rows remaining after dropping >45% missing: 5160
Columns remaining in the dataset: 101


**Analysis:**

By filtering out the rows that consist of more than 45% missing values, I removed nearly half of the dataset (4,840 rows were dropped). However, because I only performed a row-wise deletion, the structural dimensions of the dataset's features were unaffected. Therefore, exactly 101 columns remain in the dataset.

## Question 2.4: Remove columns with more than 50% missing values

In [11]:
# Calculate the fraction of missing values per column on our newly cleaned dataframe
missing_fraction_per_col = df_loan_rows_cleaned.isnull().mean()

# Keep only the columns where the fraction of missing values is 50% (0.50) or less
columns_to_keep = missing_fraction_per_col[missing_fraction_per_col <= 0.50].index
df_loan_cols_cleaned = df_loan_rows_cleaned[columns_to_keep]

# Get the new number of columns
columns_remaining = df_loan_cols_cleaned.shape[1]

print(f"Columns remaining after dropping >50% missing: {columns_remaining}")

Columns remaining after dropping >50% missing: 100


**Analysis:**

After applying the filter to remove columns that consist of more than 50% missing values (evaluated on our row-cleaned dataset), I found that 100 columns remain. This means only one single column was dropped during this step. By doing this, I've stripped out the most incomplete feature while safely retaining the vast majority of the variables for future analysis.

## Question 2.5: Replace missing values in categorical columns

In [12]:
# Create a clean copy to avoid SettingWithCopyWarning warnings
df_loan_imputed = df_loan_cols_cleaned.copy()

# Identify all categorical columns (typically 'object' data type in pandas)
categorical_cols = df_loan_imputed.select_dtypes(include=['object']).columns

# Fill missing values in these columns with the string 'Unknown'
df_loan_imputed[categorical_cols] = df_loan_imputed[categorical_cols].fillna('Unknown')

# Check how many missing values remain in the categorical columns to verify it worked
remaining_missing_cat = df_loan_imputed[categorical_cols].isnull().sum().sum()

print(f"Categorical columns processed: {len(categorical_cols)}")
print(f"Missing values remaining in categorical columns: {remaining_missing_cat}")

Categorical columns processed: 17
Missing values remaining in categorical columns: 0


**Analysis:**

By isolating the categorical columns, I found 17 features with object data types. I successfully replaced all missing values within these specific columns with the string 'Unknown'. As the output verifies, there are now zero missing values remaining across these categorical features, ensuring they are ready for encoding or further analysis without having to drop any additional rows.

##Question 2.6: Replace missing values in numeric columns

In [13]:
# Identify all numeric columns (integers and floats)
numeric_cols = df_loan_imputed.select_dtypes(include=['number']).columns

# Fill missing values in these columns with the mean of each respective column
df_loan_imputed[numeric_cols] = df_loan_imputed[numeric_cols].fillna(df_loan_imputed[numeric_cols].mean())

# Verify that no missing values remain in the entire dataset
final_missing = df_loan_imputed.isnull().sum().sum()

print(f"Numeric columns processed: {len(numeric_cols)}")
print(f"Total missing values remaining in the ENTIRE dataset: {final_missing}")

Numeric columns processed: 83
Total missing values remaining in the ENTIRE dataset: 0


**Analysis:**

For the remaining 83 numeric columns, I replaced all missing values with the respective mean of each column.

As the final output confirms, there are now completely 0 missing values remaining in the entire dataset.

By using mean imputation for the numeric data, I preserved the overall central tendency of these features without having to discard any more rows.

The dataset is now fully pre-processed, clean, and ready for any downstream machine learning models or statistical analysis.